# LeetCode 3417 — Zigzag Grid Traversal With Skip bằng Python

Link đề: https://leetcode.com/problems/zigzag-grid-traversal-with-skip/ · Độ khó: **Easy** · Chủ đề: Array, Matrix (ma trận), Simulation (mô phỏng)

Skeleton LeetCode:

```python
class Solution:
    def zigzagTraversal(self, grid: List[List[int]]) -> List[int]:
        ...
```

Ghi chú: ba ví dụ chính lấy từ đề; mọi đáp án trong notebook đã được kiểm chứng bằng code (so sánh năm cách làm với nhau và với một bộ mô phỏng toạ độ độc lập). Ví dụ nào do mình tự dựng sẽ ghi rõ "tự dựng" — nếu cần khớp từng chữ thì nên đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đi qua ma trận `grid` theo kiểu **zigzag** (hàng 0 đi từ **trái sang phải**, hàng 1 đi từ **phải sang trái**, hàng 2 lại từ trái sang phải, ...) và **bỏ qua mỗi ô thứ hai** (lấy ô thứ 1, bỏ ô thứ 2, lấy ô thứ 3, ...). Trả về giá trị các ô **được lấy**, theo đúng thứ tự đi.
2. Nắm các công cụ Python: danh sách lồng nhau `grid[i][j]`, `enumerate`, đảo danh sách (`row[::-1]`, `reversed`, `list.reverse()` và cái bẫy "đảo tại chỗ"), **cắt lát có bước** `a[::2]`, kiểm tra chẵn/lẻ bằng `% 2`, `list.extend`, `itertools.chain` và `itertools.islice`.
3. Đi từ cách **dựng cả đường đi zigzag rồi lấy phần tử cách một** → **một cờ bật/tắt khi duyệt** → **bộ đếm toàn cục `k`** → **cắt lát theo từng hàng** → **`chain` + `islice`**.
4. Hiểu mấu chốt: việc "bỏ qua cách một ô" áp dụng cho **toàn bộ đường đi liên tục**, không phải cho từng hàng. Vì thế trạng thái lấy/bỏ phải **được mang sang hàng kế tiếp** (đặc biệt khi số cột là số lẻ).
5. Tránh các lỗi kinh điển: reset cờ ở đầu mỗi hàng, quên đảo hàng lẻ, đảo hàng ngay trong `grid` làm hỏng đầu vào, và nhầm "lấy ô chỉ số chẵn của hàng" với "lấy ô chỉ số chẵn của đường đi".

## 2. Đề bài & ví dụ

Cho một ma trận `grid` kích thước `m x n` gồm các số nguyên **dương**.

Hãy duyệt `grid` theo kiểu **zigzag** nhưng **bỏ qua mỗi ô xen kẽ**:

- Bắt đầu từ ô góc trên bên trái `(0, 0)`.
- Đi **sang phải** trong hàng cho đến hết hàng.
- Xuống hàng kế tiếp rồi đi **sang trái** cho đến đầu hàng.
- Cứ **xen kẽ** phải/trái như vậy cho đến khi duyệt hết mọi hàng.

**Lưu ý**: bạn **phải bỏ qua** mỗi ô xen kẽ trong lúc duyệt (lấy ô thứ 1, bỏ ô thứ 2, lấy ô thứ 3, ...).

Trả về mảng `result` chứa **theo đúng thứ tự** giá trị của các ô **được lấy**.

**Ví dụ 1**

- Input: `grid = [[1,2],[3,4]]`
- Output: `[1,4]`
- Giải thích: đường đi zigzag đầy đủ là `1, 2, 4, 3` (hàng 0 từ trái sang phải: `1, 2`; hàng 1 từ phải sang trái: `4, 3`). Lấy ô thứ 1 và thứ 3: `1` và `4`.

**Ví dụ 2**

- Input: `grid = [[2,1],[2,1],[2,1]]`
- Output: `[2,1,2]`
- Giải thích: đường đi đầy đủ: `2, 1 | 1, 2 | 2, 1`; lấy các vị trí `0, 2, 4` ⇒ `2, 1, 2`.

**Ví dụ 3**

- Input: `grid = [[1,2,3],[4,5,6],[7,8,9]]`
- Output: `[1,3,5,7,9]`
- Giải thích: đường đi đầy đủ: `1, 2, 3 | 6, 5, 4 | 7, 8, 9`; lấy các vị trí `0, 2, 4, 6, 8` ⇒ `1, 3, 5, 7, 9`.

**Ràng buộc**

- `2 <= grid.length <= 50` và `2 <= grid[i].length <= 50`
- `1 <= grid[i][j] <= 2500`

(Đề gốc đặt tên `n == grid.length`, `m == grid[i].length`, hơi ngược thói quen. Trong notebook này mình gọi `rows` là số hàng và `cols` là số cột cho dễ đọc.)

In [1]:
from typing import List

examples = [
    ([[1, 2], [3, 4]], [1, 4]),
    ([[2, 1], [2, 1], [2, 1]], [2, 1, 2]),
    ([[1, 2, 3], [4, 5, 6], [7, 8, 9]], [1, 3, 5, 7, 9]),
]

for grid, expected in examples:
    print("grid =", grid, "-> kỳ vọng", expected)

grid = [[1, 2], [3, 4]] -> kỳ vọng [1, 4]
grid = [[2, 1], [2, 1], [2, 1]] -> kỳ vọng [2, 1, 2]
grid = [[1, 2, 3], [4, 5, 6], [7, 8, 9]] -> kỳ vọng [1, 3, 5, 7, 9]


## 3. Kiến thức nền

### 3.1. Ma trận là "danh sách của các danh sách"

Trong Python, ma trận thường là một `list` mà **mỗi phần tử là một hàng** (cũng là một `list`). `grid[i]` là hàng thứ `i`, còn `grid[i][j]` là ô ở hàng `i`, cột `j` (đếm từ `0`). `len(grid)` là số hàng, `len(grid[0])` là số cột.

In [2]:
grid = [[1, 2, 3],
        [4, 5, 6]]
print("số hàng:", len(grid))               # 2
print("số cột :", len(grid[0]))            # 3
print("hàng 1 :", grid[1])                 # [4, 5, 6]
print("ô (1,2):", grid[1][2])              # 6

số hàng: 2
số cột : 3
hàng 1 : [4, 5, 6]
ô (1,2): 6


### 3.2. `enumerate`: lấy cả chỉ số và phần tử

`for i, row in enumerate(grid):` cho ta **chỉ số hàng `i`** cùng **chính hàng đó `row`** trong mỗi vòng lặp. Ta cần `i` để biết hàng chẵn hay lẻ (quyết định hướng đi).

In [3]:
grid = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
for i, row in enumerate(grid):
    print(f"hàng {i}: {row}")

hàng 0: [1, 2, 3]
hàng 1: [4, 5, 6]
hàng 2: [7, 8, 9]


### 3.3. Đảo ngược một danh sách: ba cách, một cái bẫy

Hàng lẻ phải đi **từ phải sang trái**, tức là đọc hàng theo thứ tự **ngược**. Có ba cách:

- `row[::-1]`: tạo **danh sách mới** đã đảo (không đụng vào `row`).
- `reversed(row)`: trả về một **iterator** đi ngược (không tạo bản sao, dùng được trong `for`; muốn thành list thì `list(reversed(row))`).
- `row.reverse()`: đảo **ngay tại chỗ** và trả về `None` — **làm thay đổi `grid` của người gọi**. Nên tránh trong bài này.

In [4]:
row = [4, 5, 6]
print(row[::-1], "| row sau đó:", row)                   # [6, 5, 4] | [4, 5, 6]
print(list(reversed(row)), "| row sau đó:", row)         # [6, 5, 4] | [4, 5, 6]

r2 = [4, 5, 6]
kq = r2.reverse()
print(kq, "| r2 sau đó:", r2)                            # None | [6, 5, 4]  (r2 đã bị đổi!)

[6, 5, 4] | row sau đó: [4, 5, 6]
[6, 5, 4] | row sau đó: [4, 5, 6]
None | r2 sau đó: [6, 5, 4]


### 3.4. Cắt lát có bước: `a[bắt_đầu::bước]`

`a[0::2]` (hay `a[::2]`) lấy các phần tử ở chỉ số `0, 2, 4, ...`; `a[1::2]` lấy chỉ số `1, 3, 5, ...`. Đây chính là phép "lấy một, bỏ một" mà đề cần, áp dụng lên một dãy phẳng.

In [5]:
a = [10, 20, 30, 40, 50, 60, 70]
print(a[::2])      # [10, 30, 50, 70]  -> lấy chỉ số chẵn
print(a[1::2])     # [20, 40, 60]      -> lấy chỉ số lẻ
print(a[2::2])     # [30, 50, 70]

[10, 30, 50, 70]
[20, 40, 60]
[30, 50, 70]


### 3.5. Chẵn / lẻ bằng `% 2`

`x % 2` là phần dư khi chia `x` cho `2`: `0` nếu `x` chẵn, `1` nếu `x` lẻ. Trong `if x % 2:` thì `1` được coi là `True`, `0` là `False`. Bài này dùng `i % 2` cho **chỉ số hàng** (hàng lẻ đi ngược) và `k % 2` cho **thứ tự ô trên đường đi** (ô chỉ số chẵn được lấy).

In [6]:
for x in range(6):
    print(f"x={x}: x % 2 = {x % 2} -> {'lẻ' if x % 2 else 'chẵn'}")

x=0: x % 2 = 0 -> chẵn
x=1: x % 2 = 1 -> lẻ
x=2: x % 2 = 0 -> chẵn
x=3: x % 2 = 1 -> lẻ
x=4: x % 2 = 0 -> chẵn
x=5: x % 2 = 1 -> lẻ


### 3.6. Nối danh sách: `append`, `extend`, `itertools.chain`

- `lst.append(x)`: thêm **một** phần tử.
- `lst.extend(other)`: thêm **từng phần tử** của `other` vào cuối `lst`.
- `itertools.chain.from_iterable(list_of_lists)`: "nối đuôi" nhiều dãy thành **một dòng chảy** (iterator) mà không tạo list trung gian.
- `itertools.islice(it, start, stop, step)`: giống cắt lát nhưng dùng được trên iterator.

In [7]:
from itertools import chain, islice

flat = []
flat.extend([1, 2])
flat.extend([4, 3])
print(flat)                                              # [1, 2, 4, 3]

lists = [[1, 2], [4, 3]]
print(list(chain.from_iterable(lists)))                  # [1, 2, 4, 3]
print(list(islice(chain.from_iterable(lists), 0, None, 2)))   # [1, 4]

[1, 2, 4, 3]
[1, 2, 4, 3]
[1, 4]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Lấy `grid = [[1,2,3],[4,5,6],[7,8,9]]`. Vẽ mũi tên theo đề:

```
hàng 0:  1 -> 2 -> 3          (trái sang phải)
                    |
hàng 1:  4 <- 5 <- 6          (phải sang trái)
         |
hàng 2:  7 -> 8 -> 9          (trái sang phải)
```

**Bước 1 — viết đường đi zigzag thành một dãy phẳng:** `1, 2, 3, 6, 5, 4, 7, 8, 9`.

**Bước 2 — đánh số thứ tự từ 0 rồi lấy các vị trí chẵn:**

| vị trí `p` | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|---|---|---|---|---|---|---|---|---|---|
| giá trị | 1 | 2 | 3 | 6 | 5 | 4 | 7 | 8 | 9 |
| lấy? | có | bỏ | có | bỏ | có | bỏ | có | bỏ | có |

Kết quả: `1, 3, 5, 7, 9`.

Để ý: hàng 1 (`6, 5, 4`) bắt đầu ở vị trí `3` (lẻ) nên ô **đầu tiên** của hàng 1 (`6`) bị **bỏ**, còn `5` được lấy. Tức là **hàng sau tiếp tục "nhịp" lấy/bỏ của hàng trước**, không bắt đầu lại.

In [8]:
grid = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]

path = []
for i, row in enumerate(grid):
    path.extend(row if i % 2 == 0 else row[::-1])
print("đường đi zigzag:", path)

for p, v in enumerate(path):
    print(f"p={p} giá trị={v} -> {'lấy' if p % 2 == 0 else 'bỏ'}")

đường đi zigzag: [1, 2, 3, 6, 5, 4, 7, 8, 9]
p=0 giá trị=1 -> lấy
p=1 giá trị=2 -> bỏ
p=2 giá trị=3 -> lấy
p=3 giá trị=6 -> bỏ
p=4 giá trị=5 -> lấy
p=5 giá trị=4 -> bỏ
p=6 giá trị=7 -> lấy
p=7 giá trị=8 -> bỏ
p=8 giá trị=9 -> lấy


## 5. Cách 1 — Brute Force / baseline: dựng cả đường đi rồi lấy cách một

**Ý tưởng.** Làm đúng như tính tay:

1. Tạo danh sách `path` rỗng.
2. Với mỗi hàng `i`: hàng chẵn thì thêm các phần tử theo thứ tự thường, hàng lẻ thì thêm theo thứ tự đảo.
3. Trả về `path[::2]` (lấy các vị trí chẵn).

Cách này tách rõ hai việc: "đi zigzag" và "bỏ cách một". Rất dễ đúng, dễ debug.

In [9]:
def zigzag_skip_build_then_slice(grid: List[List[int]]) -> List[int]:
    path = []
    for i, row in enumerate(grid):
        if i % 2 == 0:
            path.extend(row)            # hàng chẵn: trái -> phải
        else:
            path.extend(row[::-1])      # hàng lẻ : phải -> trái
    return path[::2]                    # lấy vị trí 0, 2, 4, ...


for grid, expected in examples:
    print(zigzag_skip_build_then_slice(grid), "(kỳ vọng", expected, ")")

[1, 4] (kỳ vọng [1, 4] )
[2, 1, 2] (kỳ vọng [2, 1, 2] )
[1, 3, 5, 7, 9] (kỳ vọng [1, 3, 5, 7, 9] )


**Độ phức tạp (Cách 1).** Gọi `N = rows × cols` là tổng số ô. Dựng `path` mất `O(N)` thời gian, `path[::2]` thêm `O(N)` ⇒ **Time `O(N)`**. Bộ nhớ: `path` chứa `N` phần tử và kết quả `N/2` ⇒ **Space `O(N)`**.

**Nút thắt:** thời gian đã tối ưu (phải nhìn mọi ô ít nhất một lần). Chỉ có thể cải thiện **bộ nhớ phụ**: ta chưa cần lưu hết đường đi, chỉ cần biết ô hiện tại **được lấy hay bỏ**.

## 6. Ý tưởng thuật toán / bước đột phá

Đường đi zigzag là một **dãy liên tục**; ô ở vị trí `p` (đếm từ `0`) được lấy khi `p` **chẵn**. Vậy ta chỉ cần:

- đi theo thứ tự zigzag (hàng lẻ đọc ngược), và
- giữ **một biến đếm** (hoặc **một cờ** bật/tắt) cho biết ô hiện tại thuộc "nhịp lấy" hay "nhịp bỏ".

Cạm bẫy lớn nhất: **cờ/bộ đếm phải được giữ nguyên khi sang hàng mới.** Nếu số cột `cols` là số lẻ thì hàng kế tiếp bắt đầu ở nhịp **ngược** với hàng trước. Nếu số cột chẵn thì mỗi hàng đều bắt đầu ở nhịp "lấy". Ta có công thức: vị trí của ô đầu hàng `i` là `p = i × cols`, nên ô đầu hàng được lấy khi `i × cols` chẵn.

In [10]:
# Ma trận 3 hàng x 3 cột (cols lẻ): nhịp bắt đầu mỗi hàng luân phiên
# Ma trận 3 hàng x 2 cột (cols chẵn): mọi hàng đều bắt đầu ở nhịp "lấy"
for rows, cols in [(3, 3), (3, 2), (4, 5)]:
    print(f"rows={rows}, cols={cols}")
    for i in range(rows):
        p_dau = i * cols
        print(f"  hàng {i}: vị trí đầu p={p_dau} -> ô đầu hàng {'được lấy' if p_dau % 2 == 0 else 'bị bỏ'}")

rows=3, cols=3
  hàng 0: vị trí đầu p=0 -> ô đầu hàng được lấy
  hàng 1: vị trí đầu p=3 -> ô đầu hàng bị bỏ
  hàng 2: vị trí đầu p=6 -> ô đầu hàng được lấy
rows=3, cols=2
  hàng 0: vị trí đầu p=0 -> ô đầu hàng được lấy
  hàng 1: vị trí đầu p=2 -> ô đầu hàng được lấy
  hàng 2: vị trí đầu p=4 -> ô đầu hàng được lấy
rows=4, cols=5
  hàng 0: vị trí đầu p=0 -> ô đầu hàng được lấy
  hàng 1: vị trí đầu p=5 -> ô đầu hàng bị bỏ
  hàng 2: vị trí đầu p=10 -> ô đầu hàng được lấy
  hàng 3: vị trí đầu p=15 -> ô đầu hàng bị bỏ


## 7. Các cách tiếp theo

### Cách 2 — Một cờ `take` bật/tắt khi duyệt (không lưu đường đi)

**Giải thích từng đoạn.**

1. `take = True`: ô đầu tiên của đường đi được lấy.
2. Với mỗi hàng `i`: nếu `i` lẻ thì đọc ngược (`row[::-1]`), ngược lại đọc thường.
3. Với mỗi giá trị `x` trong hàng đã sắp thứ tự: nếu `take` là `True` thì thêm `x` vào `ans`.
4. Sau mỗi ô, **đảo cờ**: `take = not take`. Cờ **không bị reset** khi sang hàng mới.

In [11]:
def zigzag_skip_flag(grid: List[List[int]]) -> List[int]:
    ans = []
    take = True                                  # ô đầu tiên được lấy
    for i, row in enumerate(grid):
        ordered = row if i % 2 == 0 else row[::-1]
        for x in ordered:
            if take:
                ans.append(x)
            take = not take                      # lấy xong thì bỏ, bỏ xong thì lấy
    return ans


for grid, expected in examples:
    print(zigzag_skip_flag(grid), "(kỳ vọng", expected, ")")

[1, 4] (kỳ vọng [1, 4] )
[2, 1, 2] (kỳ vọng [2, 1, 2] )
[1, 3, 5, 7, 9] (kỳ vọng [1, 3, 5, 7, 9] )


In [12]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionFlag:
    def zigzagTraversal(self, grid: List[List[int]]) -> List[int]:
        ans = []
        take = True
        for i, row in enumerate(grid):
            ordered = row if i % 2 == 0 else row[::-1]
            for x in ordered:
                if take:
                    ans.append(x)
                take = not take
        return ans


print(SolutionFlag().zigzagTraversal([[1, 2, 3], [4, 5, 6], [7, 8, 9]]))     # [1, 3, 5, 7, 9]

[1, 3, 5, 7, 9]


**Độ phức tạp Cách 2:** **Time `O(N)`** (`N = rows × cols`). **Space `O(1)` phụ** ngoài kết quả (`row[::-1]` tạo bản sao một hàng, tức `O(cols)` tạm thời; nếu muốn thật sự `O(1)` có thể duyệt chỉ số ngược).

### Cách 3 — Bộ đếm toàn cục `k`: lấy khi `k` chẵn

**Giải thích.** Thay cờ bằng biến đếm `k` = số ô đã đi qua. Ô hiện tại được lấy khi `k % 2 == 0`. Sau mỗi ô tăng `k += 1`. Ưu điểm: `k` chính là vị trí `p` trong đường đi nên dễ liên hệ với bảng tính tay ở mục 4.

In [13]:
def zigzag_skip_counter(grid: List[List[int]]) -> List[int]:
    ans = []
    k = 0                                        # k = vị trí của ô trên đường đi
    for i, row in enumerate(grid):
        ordered = row if i % 2 == 0 else reversed(row)
        for x in ordered:
            if k % 2 == 0:
                ans.append(x)
            k += 1
    return ans


for grid, expected in examples:
    print(zigzag_skip_counter(grid), "(kỳ vọng", expected, ")")

[1, 4] (kỳ vọng [1, 4] )
[2, 1, 2] (kỳ vọng [2, 1, 2] )
[1, 3, 5, 7, 9] (kỳ vọng [1, 3, 5, 7, 9] )


In [14]:
# Code nộp LeetCode cho Cách 3
class SolutionCounter:
    def zigzagTraversal(self, grid: List[List[int]]) -> List[int]:
        ans = []
        k = 0
        for i, row in enumerate(grid):
            for x in (row if i % 2 == 0 else reversed(row)):
                if k % 2 == 0:
                    ans.append(x)
                k += 1
        return ans


print(SolutionCounter().zigzagTraversal([[2, 1], [2, 1], [2, 1]]))        # [2, 1, 2]

[2, 1, 2]


**Độ phức tạp Cách 3:** **Time `O(N)`**, **Space `O(1)` phụ** (dùng `reversed` nên không sao chép hàng).

### Cách 4 — Cắt lát từng hàng, chọn điểm bắt đầu theo `k % 2`

**Giải thích.** Thay vì xét từng ô, ta xử lý **cả hàng một lần**: hàng đã sắp thứ tự (`ordered`) có `cols` ô. Nếu ô đầu hàng nằm ở vị trí chẵn (`k` chẵn) thì lấy `ordered[0::2]`, ngược lại lấy `ordered[1::2]`. Sau hàng, `k += cols`.

In [15]:
def zigzag_skip_row_slices(grid: List[List[int]]) -> List[int]:
    ans = []
    k = 0                                        # số ô đã đi qua trước hàng hiện tại
    for i, row in enumerate(grid):
        ordered = row if i % 2 == 0 else row[::-1]
        start = 0 if k % 2 == 0 else 1           # ô đầu hàng được lấy (0) hay bỏ (1)
        ans.extend(ordered[start::2])
        k += len(row)
    return ans


for grid, expected in examples:
    print(zigzag_skip_row_slices(grid), "(kỳ vọng", expected, ")")

[1, 4] (kỳ vọng [1, 4] )
[2, 1, 2] (kỳ vọng [2, 1, 2] )
[1, 3, 5, 7, 9] (kỳ vọng [1, 3, 5, 7, 9] )


In [16]:
# Code nộp LeetCode cho Cách 4
class SolutionRowSlices:
    def zigzagTraversal(self, grid: List[List[int]]) -> List[int]:
        ans = []
        k = 0
        for i, row in enumerate(grid):
            ordered = row if i % 2 == 0 else row[::-1]
            ans.extend(ordered[k % 2::2])        # k % 2 chính là điểm bắt đầu (0 hoặc 1)
            k += len(row)
        return ans


print(SolutionRowSlices().zigzagTraversal([[1, 2], [3, 4]]))              # [1, 4]

[1, 4]


**Độ phức tạp Cách 4:** **Time `O(N)`**, **Space `O(cols)`** tạm thời cho mỗi hàng. Ít lệnh `if` hơn vì mỗi hàng chỉ quyết định một lần.

### Cách 5 — `chain` + `islice`: "dòng chảy" zigzag rồi lấy cách một

**Giải thích.** Tạo một generator trả các hàng đã sắp thứ tự, nối chúng bằng `chain.from_iterable`, rồi dùng `islice(dòng, 0, None, 2)` để lấy cách một ô. Không có biến phụ, rất gọn — nhưng người mới nên hiểu Cách 1–4 trước.

In [17]:
from itertools import chain, islice


def zigzag_skip_chain(grid: List[List[int]]) -> List[int]:
    rows_in_order = (row if i % 2 == 0 else row[::-1] for i, row in enumerate(grid))
    return list(islice(chain.from_iterable(rows_in_order), 0, None, 2))


for grid, expected in examples:
    print(zigzag_skip_chain(grid), "(kỳ vọng", expected, ")")

[1, 4] (kỳ vọng [1, 4] )
[2, 1, 2] (kỳ vọng [2, 1, 2] )
[1, 3, 5, 7, 9] (kỳ vọng [1, 3, 5, 7, 9] )


In [18]:
# Code nộp LeetCode cho Cách 5
from itertools import chain, islice


class SolutionChain:
    def zigzagTraversal(self, grid: List[List[int]]) -> List[int]:
        flow = chain.from_iterable(row if i % 2 == 0 else row[::-1] for i, row in enumerate(grid))
        return list(islice(flow, 0, None, 2))


print(SolutionChain().zigzagTraversal([[1, 2, 3], [4, 5, 6], [7, 8, 9]]))   # [1, 3, 5, 7, 9]

[1, 3, 5, 7, 9]


**Độ phức tạp Cách 5:** **Time `O(N)`**, **Space `O(1)` phụ** ngoài kết quả (và bản sao một hàng khi đảo).

Tóm lại, cả năm cách đều `O(N)` thời gian; khác nhau ở độ gọn và bộ nhớ phụ. Bản nộp mình chọn là **Cách 3** (bộ đếm `k`): dễ đọc nhất và bám sát công thức "lấy khi vị trí chẵn".

## 8. TRACE từng bước

Trace **Cách 3** (bộ đếm `k`) với `grid = [[1,2,3],[4,5,6],[7,8,9]]`. Trong bảng, "hàng đã sắp" là hàng sau khi đảo (nếu cần).

| hàng `i` | hàng đã sắp | ô `x` | `k` trước | `k % 2` | quyết định | `ans` sau |
|---|---|---|---|---|---|---|
| 0 | [1,2,3] | 1 | 0 | 0 | lấy | [1] |
| 0 | [1,2,3] | 2 | 1 | 1 | bỏ | [1] |
| 0 | [1,2,3] | 3 | 2 | 0 | lấy | [1,3] |
| 1 | [6,5,4] | 6 | 3 | 1 | bỏ | [1,3] |
| 1 | [6,5,4] | 5 | 4 | 0 | lấy | [1,3,5] |
| 1 | [6,5,4] | 4 | 5 | 1 | bỏ | [1,3,5] |
| 2 | [7,8,9] | 7 | 6 | 0 | lấy | [1,3,5,7] |
| 2 | [7,8,9] | 8 | 7 | 1 | bỏ | [1,3,5,7] |
| 2 | [7,8,9] | 9 | 8 | 0 | lấy | [1,3,5,7,9] |

Hàm dưới đây in đúng các cột đó.

In [19]:
def trace_counter(grid: List[List[int]]) -> List[int]:
    ans = []
    k = 0
    for i, row in enumerate(grid):
        ordered = row if i % 2 == 0 else row[::-1]
        print(f"--- hàng {i}: {row} -> đọc theo thứ tự {ordered}")
        for x in ordered:
            quyet_dinh = "lấy" if k % 2 == 0 else "bỏ"
            if k % 2 == 0:
                ans.append(x)
            print(f"  x={x}, k={k}, k%2={k % 2} -> {quyet_dinh}, ans={ans}")
            k += 1
    return ans


assert trace_counter([[1, 2, 3], [4, 5, 6], [7, 8, 9]]) == [1, 3, 5, 7, 9]

--- hàng 0: [1, 2, 3] -> đọc theo thứ tự [1, 2, 3]
  x=1, k=0, k%2=0 -> lấy, ans=[1]
  x=2, k=1, k%2=1 -> bỏ, ans=[1]
  x=3, k=2, k%2=0 -> lấy, ans=[1, 3]
--- hàng 1: [4, 5, 6] -> đọc theo thứ tự [6, 5, 4]
  x=6, k=3, k%2=1 -> bỏ, ans=[1, 3]
  x=5, k=4, k%2=0 -> lấy, ans=[1, 3, 5]
  x=4, k=5, k%2=1 -> bỏ, ans=[1, 3, 5]
--- hàng 2: [7, 8, 9] -> đọc theo thứ tự [7, 8, 9]
  x=7, k=6, k%2=0 -> lấy, ans=[1, 3, 5, 7]
  x=8, k=7, k%2=1 -> bỏ, ans=[1, 3, 5, 7]
  x=9, k=8, k%2=0 -> lấy, ans=[1, 3, 5, 7, 9]


Bây giờ trace **Cách 4** (cắt lát từng hàng) trên `grid = [[2,1],[2,1],[2,1]]` (`cols = 2` chẵn: mọi hàng bắt đầu ở nhịp "lấy") và `grid = [[1,2,3],[4,5,6]]` (`cols = 3` lẻ: nhịp đổi giữa các hàng).

In [20]:
def trace_row_slices(grid: List[List[int]]) -> List[int]:
    ans = []
    k = 0
    for i, row in enumerate(grid):
        ordered = row if i % 2 == 0 else row[::-1]
        start = k % 2
        chosen = ordered[start::2]
        print(f"hàng {i}: ordered={ordered}, k={k}, start={start}, lấy {chosen}")
        ans.extend(chosen)
        k += len(row)
    print("kết quả:", ans)
    return ans


assert trace_row_slices([[2, 1], [2, 1], [2, 1]]) == [2, 1, 2]
print()
assert trace_row_slices([[1, 2, 3], [4, 5, 6]]) == [1, 3, 5]

hàng 0: ordered=[2, 1], k=0, start=0, lấy [2]
hàng 1: ordered=[1, 2], k=2, start=0, lấy [1]
hàng 2: ordered=[2, 1], k=4, start=0, lấy [2]
kết quả: [2, 1, 2]

hàng 0: ordered=[1, 2, 3], k=0, start=0, lấy [1, 3]
hàng 1: ordered=[6, 5, 4], k=3, start=1, lấy [5]
kết quả: [1, 3, 5]


## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (bộ đếm `k` + `reversed`). `List` đã được LeetCode import sẵn; ở notebook mình đã `from typing import List` ở trên.

In [21]:
class Solution:
    def zigzagTraversal(self, grid: List[List[int]]) -> List[int]:
        ans = []
        k = 0                                          # vị trí của ô hiện tại trên đường đi zigzag
        for i, row in enumerate(grid):
            ordered = row if i % 2 == 0 else reversed(row)     # hàng lẻ đi từ phải sang trái
            for x in ordered:
                if k % 2 == 0:                         # chỉ lấy các ô ở vị trí chẵn
                    ans.append(x)
                k += 1
        return ans


sol = Solution()
for grid, expected in examples:
    print(sol.zigzagTraversal(grid), "(kỳ vọng", expected, ")")

[1, 4] (kỳ vọng [1, 4] )
[2, 1, 2] (kỳ vọng [2, 1, 2] )
[1, 3, 5, 7, 9] (kỳ vọng [1, 3, 5, 7, 9] )


## 10. Vì sao thuật toán đúng

- **Thứ tự duyệt đúng:** hàng `i` chẵn đọc từ cột `0` đến cột cuối; hàng `i` lẻ đọc ngược. Đó đúng là "sang phải, xuống, sang trái, xuống, ..." của đề.
- **Đếm toàn cục:** `k` tăng sau **mỗi** ô và **không bị reset** giữa các hàng, nên `k` luôn bằng số ô đã đi qua, tức vị trí của ô hiện tại trên đường đi liên tục.
- **Bỏ qua xen kẽ đúng nghĩa:** đề lấy ô thứ 1, bỏ ô thứ 2, ... tức lấy các ô có vị trí (đếm từ `0`) **chẵn**. Điều kiện `k % 2 == 0` chính là điều kiện đó.
- **Thứ tự kết quả:** ta `append` theo đúng thứ tự gặp ô nên `ans` đã đúng thứ tự cần trả.
- **Kích thước:** có `N = rows × cols` ô, lấy các vị trí `0, 2, ...` nên `len(ans) = ceil(N / 2)` (làm tròn lên). Có thể dùng làm kiểm tra nhanh ở mục test.

## 11. Phân tích Time / Space Complexity

Gọi `N = rows × cols` (tối đa `50 × 50 = 2500` ô).

| Cách | Time | Space phụ (ngoài kết quả) |
|---|---|---|
| 1. Dựng đường đi rồi `[::2]` | `O(N)` | `O(N)` |
| 2. Cờ `take` | `O(N)` | `O(cols)` (bản sao hàng đảo) |
| 3. Bộ đếm `k` + `reversed` | `O(N)` | `O(1)` |
| 4. Cắt lát từng hàng | `O(N)` | `O(cols)` |
| 5. `chain` + `islice` | `O(N)` | `O(cols)` |

Không thể nhanh hơn `O(N)` về lý thuyết nếu phải đọc mọi ô; thực ra ta chỉ **chạm** nửa số ô khi dùng cắt lát, nhưng vẫn phải dịch chuyển theo `N` ô nên độ phức tạp tiệm cận không đổi.

## 12. Bộ test

Ta gom tất cả các cách vào `impls` rồi so với kỳ vọng. Ngoài ra có một **bộ mô phỏng toạ độ độc lập** (`oracle_coords`) dùng vòng `while` với `(r, c)` và hướng đi — viết theo cách khác hẳn để làm "trọng tài".

In [22]:
import copy
import random

sol = Solution()
impls = {
    "build_then_slice": zigzag_skip_build_then_slice,
    "flag": zigzag_skip_flag,
    "counter": zigzag_skip_counter,
    "row_slices": zigzag_skip_row_slices,
    "chain": zigzag_skip_chain,
    "leetcode": sol.zigzagTraversal,
}


def oracle_coords(grid):
    """Mô phỏng bằng toạ độ: đi từng bước, đổi hàng khi chạm mép."""
    rows, cols = len(grid), len(grid[0])
    r, c, step = 0, 0, 1                         # step = +1 (sang phải) hoặc -1 (sang trái)
    visited = []
    while r < rows:
        visited.append(grid[r][c])
        if 0 <= c + step < cols:
            c += step
        else:
            r += 1
            step = -step
    return [v for idx, v in enumerate(visited) if idx % 2 == 0]


def check_all(grid, expected, label):
    for key, fn in impls.items():
        actual = fn(copy.deepcopy(grid))
        assert actual == expected, (label, key, grid, actual, expected)
    assert oracle_coords(grid) == expected, (label, "oracle")
    print("PASS", label)


for idx, (grid, expected) in enumerate(examples, 1):
    check_all(grid, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3


### 12.1. Các trường hợp biên

Ràng buộc đảm bảo ít nhất `2 × 2`, nhưng ta cũng thử vài hình dạng: nhiều hàng, nhiều cột, cột chẵn/lẻ, và kích thước lớn nhất `50 × 50`. Các ví dụ "tự dựng" dưới đây có kỳ vọng được tính tay (hoặc bằng công thức) ngay trong comment.

In [23]:
# 2 x 2 nhỏ nhất
check_all([[5, 6], [7, 8]], [5, 8], "2x2: 5,6,8,7 -> 5,8")
# 2 hàng, 3 cột (cols lẻ): đường đi 1,2,3,6,5,4 -> lấy 1,3,5
check_all([[1, 2, 3], [4, 5, 6]], [1, 3, 5], "2x3")
# 4 hàng, 2 cột (cols chẵn): đường đi 1,2,4,3,5,6,8,7 -> lấy 1,4,5,8
check_all([[1, 2], [3, 4], [5, 6], [7, 8]], [1, 4, 5, 8], "4x2")
# 3 hàng, 4 cột: đường đi 1,2,3,4,8,7,6,5,9,10,11,12 -> lấy 1,3,8,6,9,11
check_all([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]], [1, 3, 8, 6, 9, 11], "3x4")
# toàn số giống nhau: kết quả chỉ phụ thuộc số lượng = ceil(N/2)
check_all([[7] * 5 for _ in range(5)], [7] * 13, "5x5 toàn 7 (25 ô -> 13)")
# kích thước lớn nhất
big = [[(i * 50 + j) % 2500 + 1 for j in range(50)] for i in range(50)]
assert len(sol.zigzagTraversal(big)) == 1250
check_all(big, oracle_coords(big), "50x50")

PASS 2x2: 5,6,8,7 -> 5,8
PASS 2x3
PASS 4x2
PASS 3x4
PASS 5x5 toàn 7 (25 ô -> 13)
PASS 50x50


### 12.2. Phản ví dụ cho các cách làm sai

Hai cách "nghe có vẻ đúng" nhưng sai: (a) **reset cờ ở đầu mỗi hàng**; (b) **lấy chỉ số cột chẵn trong mỗi hàng** (không xét hướng đi và nhịp toàn cục). Ví dụ 3 (`3 × 3`) đủ để lộ cả hai.

In [24]:
def wrong_reset_each_row(grid):
    ans = []
    for i, row in enumerate(grid):
        take = True                                # SAI: reset ở mỗi hàng
        for x in (row if i % 2 == 0 else row[::-1]):
            if take:
                ans.append(x)
            take = not take
    return ans


def wrong_even_columns(grid):
    ans = []
    for row in grid:
        ans.extend(row[::2])                       # SAI: lấy cột chẵn, không đảo, không nhịp toàn cục
    return ans


g = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
print(wrong_reset_each_row(g), "!=", sol.zigzagTraversal(g))
print(wrong_even_columns(g), "!=", sol.zigzagTraversal(g))
assert wrong_reset_each_row(g) != sol.zigzagTraversal(g)
assert wrong_even_columns([[1, 2], [3, 4]]) != sol.zigzagTraversal([[1, 2], [3, 4]])

[1, 3, 6, 4, 7, 9] != [1, 3, 5, 7, 9]
[1, 3, 4, 6, 7, 9] != [1, 3, 5, 7, 9]


### 12.3. Đối chiếu ngẫu nhiên (có seed) và tính chất

Sinh ngẫu nhiên các ma trận `rows, cols` trong `[2, 50]`, giá trị trong `[1, 2500]` (đúng ràng buộc). Kiểm tra: (1) năm cách + bản nộp + `oracle_coords` cho **cùng kết quả**; (2) độ dài kết quả bằng `(rows × cols + 1) // 2`; (3) **đầu vào không bị thay đổi**.

In [25]:
rng = random.Random(3417)
for trial in range(1500):
    rows = rng.randint(2, 50)
    cols = rng.randint(2, 50)
    grid = [[rng.randint(1, 2500) for _ in range(cols)] for _ in range(rows)]
    snapshot = copy.deepcopy(grid)
    ref = oracle_coords(grid)
    for key, fn in impls.items():
        assert fn(grid) == ref, (key, rows, cols)
        assert grid == snapshot, ("đầu vào bị thay đổi bởi", key)
    assert len(ref) == (rows * cols + 1) // 2
print("PASS 1500 ma trận ngẫu nhiên đúng ràng buộc")

PASS 1500 ma trận ngẫu nhiên đúng ràng buộc


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Reset cờ ở đầu mỗi hàng

In [26]:
def sai_reset_co(grid):
    ans = []
    for i, row in enumerate(grid):
        take = True                          # SAI: hàng nào cũng bắt đầu bằng "lấy"
        for x in (row if i % 2 == 0 else row[::-1]):
            if take:
                ans.append(x)
            take = not take
    return ans


g = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
print("sai:", sai_reset_co(g), "| đúng:", Solution().zigzagTraversal(g))

sai: [1, 3, 6, 4, 7, 9] | đúng: [1, 3, 5, 7, 9]


**Vì sao sai:** đề bỏ qua ô xen kẽ **trên cả đường đi**. Khi số cột là số lẻ (như `3`), hàng 0 kết thúc ở nhịp "lấy" nên ô đầu của hàng 1 phải là "bỏ". Reset cờ làm hàng 1 lấy `6` và `4` thay vì `5`. **Cách sửa:** khai báo `take` (hoặc `k`) **ngoài** vòng lặp hàng.

### Lỗi 2: Quên đảo hàng lẻ

In [27]:
def sai_quen_dao(grid):
    ans = []
    k = 0
    for i, row in enumerate(grid):
        for x in row:                        # SAI: không đảo hàng lẻ
            if k % 2 == 0:
                ans.append(x)
            k += 1
    return ans


g = [[1, 2], [3, 4]]
print("sai:", sai_quen_dao(g), "| đúng:", Solution().zigzagTraversal(g))

sai: [1, 3] | đúng: [1, 4]


**Vì sao sai:** nếu hàng 1 đọc `3, 4` (trái sang phải) thì đường đi thành `1, 2, 3, 4` và kết quả `[1, 3]`; đúng phải là `[1, 4]` vì hàng 1 đọc `4, 3`. **Cách sửa:** `row if i % 2 == 0 else row[::-1]`.

### Lỗi 3: Đảo hàng tại chỗ bằng `row.reverse()` (làm hỏng đầu vào)

In [28]:
def sai_reverse_tai_cho(grid):
    ans = []
    k = 0
    for i, row in enumerate(grid):
        if i % 2 == 1:
            row.reverse()                    # SAI: đổi luôn grid của người gọi
        for x in row:
            if k % 2 == 0:
                ans.append(x)
            k += 1
    return ans


g = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
print("kết quả:", sai_reverse_tai_cho(g), "(vẫn đúng)")
print("nhưng grid đã bị đổi:", g)

kết quả: [1, 3, 5, 7, 9] (vẫn đúng)
nhưng grid đã bị đổi: [[1, 2, 3], [6, 5, 4], [7, 8, 9]]


**Vì sao sai:** kết quả trả về có thể vẫn đúng, nhưng hàng `[4,5,6]` đã bị đảo thành `[6,5,4]` trong `grid` gốc. Nếu gọi hàm lần thứ hai trên cùng `g`, hàng lẻ bị đảo **lần nữa** về thứ tự cũ và kết quả **sai**. **Cách sửa:** dùng `row[::-1]` hoặc `reversed(row)` (không sửa `grid`).

### Lỗi 4: Nhầm "ô chỉ số cột chẵn" với "ô vị trí chẵn trên đường đi"

In [29]:
def sai_cot_chan(grid):
    ans = []
    for i, row in enumerate(grid):
        ordered = row if i % 2 == 0 else row[::-1]
        for j, x in enumerate(ordered):
            if j % 2 == 0:                   # SAI: chỉ số trong hàng, không phải vị trí toàn đường đi
                ans.append(x)
    return ans


g = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
print("sai:", sai_cot_chan(g), "| đúng:", Solution().zigzagTraversal(g))

sai: [1, 3, 6, 4, 7, 9] | đúng: [1, 3, 5, 7, 9]


**Vì sao sai:** `j` đếm lại từ `0` ở mỗi hàng nên đây chính là lỗi reset (Lỗi 1) trá hình. Với `cols` chẵn kết quả có thể vẫn đúng, với `cols` lẻ thì sai — rất dễ lọt qua nếu chỉ thử ví dụ 1 và 2. **Cách sửa:** dùng bộ đếm toàn cục `k`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Đi zigzag / rắn / boustrophedon**" ⇒ hàng chẵn đọc thuận, hàng lẻ đọc ngược: `row if i % 2 == 0 else row[::-1]`.
- "**Bỏ qua mỗi phần tử xen kẽ**" trên một dãy liên tục ⇒ **một bộ đếm toàn cục** hoặc cắt lát `[::2]`. Đừng đặt lại nhịp khi sang hàng mới.
- Khi chưa chắc, **làm phẳng đường đi ra một list rồi lấy `[::2]`** (Cách 1). Sau đó mới tối ưu.
- Kiểm tra nhanh bằng độ dài: `len(ans) == (rows * cols + 1) // 2`.
- Kiểm tra **hai loại hình dạng**: số cột chẵn và số cột lẻ. Lỗi về "nhịp" chỉ lộ ra khi số cột lẻ.
- Dạng bài mô phỏng ma trận khác (xoắn ốc, đường chéo): luôn phân biệt "**thứ tự duyệt**" và "**điều kiện chọn**" làm hai bước.

## 15. Biến thể của bài

1. **Lấy cách `k` ô** (không chỉ cách một): lấy khi `p % k == 0`.
2. **Bắt đầu bỏ trước** (bỏ ô đầu, lấy ô thứ hai, ...): đổi điều kiện thành `p % 2 == 1`.
3. **Zigzag theo cột** thay vì theo hàng (cột chẵn đi xuống, cột lẻ đi lên).
4. **Trả về toạ độ** `(i, j)` của các ô được lấy thay vì giá trị.
5. **Duyệt xoắn ốc** (LeetCode 54 Spiral Matrix) hoặc **Diagonal Traverse** (LeetCode 498): cùng ý tưởng "thứ tự duyệt" + "điều kiện chọn".

In [30]:
def zigzag_every_kth(grid, k):
    """Biến thể 1: lấy ô có vị trí p chia hết cho k."""
    ans, p = [], 0
    for i, row in enumerate(grid):
        for x in (row if i % 2 == 0 else reversed(row)):
            if p % k == 0:
                ans.append(x)
            p += 1
    return ans


def zigzag_start_skipped(grid):
    """Biến thể 2: bỏ ô đầu, lấy ô thứ hai, bỏ ô thứ ba, ..."""
    ans, p = [], 0
    for i, row in enumerate(grid):
        for x in (row if i % 2 == 0 else reversed(row)):
            if p % 2 == 1:
                ans.append(x)
            p += 1
    return ans


def zigzag_coordinates(grid):
    """Biến thể 4: trả toạ độ (i, j) các ô được lấy."""
    ans, p = [], 0
    for i, row in enumerate(grid):
        cols_in_order = range(len(row)) if i % 2 == 0 else range(len(row) - 1, -1, -1)
        for j in cols_in_order:
            if p % 2 == 0:
                ans.append((i, j))
            p += 1
    return ans


g = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
print(zigzag_every_kth(g, 3))        # vị trí 0,3,6 trên đường 1,2,3,6,5,4,7,8,9 -> [1, 6, 7]
print(zigzag_start_skipped(g))       # vị trí 1,3,5,7 -> [2, 6, 4, 8]
print(zigzag_coordinates(g))
assert zigzag_every_kth(g, 3) == [1, 6, 7]
assert zigzag_start_skipped(g) == [2, 6, 4, 8]
assert [g[i][j] for i, j in zigzag_coordinates(g)] == Solution().zigzagTraversal(g)

[1, 6, 7]
[2, 6, 4, 8]
[(0, 0), (0, 2), (1, 1), (2, 0), (2, 2)]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space phụ | Khi nào dùng |
|---|---|---|---|---|
| 1. Dựng đường đi + `[::2]` | tách "đi" và "bỏ" thành hai bước | `O(N)` | `O(N)` | học, debug, mới làm quen |
| 2. Cờ `take` | bật/tắt sau mỗi ô | `O(N)` | `O(cols)` | trực quan, ít biến |
| 3. Bộ đếm `k` + `reversed` | lấy khi `k` chẵn | `O(N)` | `O(1)` | **nên nộp**: rõ ràng, nhẹ bộ nhớ |
| 4. Cắt lát từng hàng | `ordered[k % 2::2]` | `O(N)` | `O(cols)` | khi muốn ít `if` và tận dụng slicing |
| 5. `chain` + `islice` | dòng chảy rồi lấy cách một | `O(N)` | `O(cols)` | viết gọn, quen `itertools` |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `[10, 20, 30, 40, 50][::2]`, `[10, 20, 30, 40, 50][1::2]`, `[1, 2, 3][::-1]`, `Solution().zigzagTraversal([[1, 2], [3, 4]])`, `Solution().zigzagTraversal([[1, 2, 3], [4, 5, 6]])`.

**Bài 2 (tính tay).** Với `grid = [[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]`: viết đường đi zigzag đầy đủ, đánh số vị trí rồi suy ra kết quả. (Đây là ví dụ `3 × 4`, cột chẵn.)

**Bài 3 (điền TODO).** Hoàn thành `zigzag_todo(grid)` theo Cách 3 (bộ đếm `k`).

In [31]:
def zigzag_todo(grid):
    ans = []
    # TODO 1: khai báo k = 0 (NGOÀI vòng lặp hàng)
    for i, row in enumerate(grid):
        # TODO 2: ordered = row nếu i chẵn, ngược lại row[::-1]
        # TODO 3: duyệt từng x trong ordered; nếu k % 2 == 0 thì ans.append(x); rồi k += 1
        pass
    return ans

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `zigzag_skip_from_scratch(grid)` theo **Cách 1** nhưng **không dùng** `row[::-1]` hay `reversed`: dùng `range(len(row) - 1, -1, -1)` để đọc hàng lẻ từ cuối về đầu, rồi lấy `path[::2]`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `zigzag_skip_columns(grid)`: đi zigzag **theo cột** (cột `0` đi từ trên xuống, cột `1` đi từ dưới lên, ...), bỏ qua ô xen kẽ. (b) Viết `zigzag_sum_taken(grid)` trả về **tổng** các ô được lấy **mà không tạo list kết quả**.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[10, 30, 50]`, `[20, 40]`, `[3, 2, 1]`, `[1, 4]`, `[1, 3, 5]`. Bài 3: nhớ đặt `k = 0` **trước** vòng `for i, row`.
- **Mức 2 (vừa):** Bài 2: đường đi `1,2,3,4,8,7,6,5,9,10,11,12` (hàng 1 đảo). Vị trí chẵn: `p = 0, 2, 4, 6, 8, 10` ⇒ `1, 3, 8, 6, 9, 11`. Bài 4: `for j in range(len(row) - 1, -1, -1): path.append(row[j])`.
- **Mức 3 (sâu):** Bài 5(a): chuyển vị trước: `cols_of_grid = list(zip(*grid))` rồi áp dụng chính hàm zigzag theo hàng trên các "hàng" mới (mỗi cột thành một hàng). 5(b): thay `ans.append(x)` bằng `total += x`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert [10, 20, 30, 40, 50][::2] == [10, 30, 50]
assert [10, 20, 30, 40, 50][1::2] == [20, 40]
assert [1, 2, 3][::-1] == [3, 2, 1]
assert Solution().zigzagTraversal([[1, 2], [3, 4]]) == [1, 4]
assert Solution().zigzagTraversal([[1, 2, 3], [4, 5, 6]]) == [1, 3, 5]

# Bài 2
g2 = [[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]
assert Solution().zigzagTraversal(g2) == [1, 3, 8, 6, 9, 11]


# Bài 3
def zigzag_todo_ans(grid):
    ans = []
    k = 0
    for i, row in enumerate(grid):
        ordered = row if i % 2 == 0 else row[::-1]
        for x in ordered:
            if k % 2 == 0:
                ans.append(x)
            k += 1
    return ans


for grid, expected in examples:
    assert zigzag_todo_ans(grid) == expected


# Bài 4
def zigzag_skip_from_scratch(grid):
    path = []
    for i, row in enumerate(grid):
        if i % 2 == 0:
            for j in range(len(row)):
                path.append(row[j])
        else:
            for j in range(len(row) - 1, -1, -1):
                path.append(row[j])
    return path[::2]


for grid, expected in examples:
    assert zigzag_skip_from_scratch(grid) == expected


# Bài 5
def zigzag_skip_columns(grid):
    columns = [list(col) for col in zip(*grid)]        # mỗi cột thành một "hàng"
    return Solution().zigzagTraversal(columns)


def zigzag_sum_taken(grid):
    total, k = 0, 0
    for i, row in enumerate(grid):
        for x in (row if i % 2 == 0 else reversed(row)):
            if k % 2 == 0:
                total += x
            k += 1
    return total


# cột 0 đi xuống: 1,4,7 ; cột 1 đi lên: 8,5,2 ; cột 2 đi xuống: 3,6,9 -> đường 1,4,7,8,5,2,3,6,9 -> lấy 1,7,5,3,9
assert zigzag_skip_columns([[1, 2, 3], [4, 5, 6], [7, 8, 9]]) == [1, 7, 5, 3, 9]
for grid, expected in examples:
    assert zigzag_sum_taken(grid) == sum(expected)
print("Tất cả đáp án đều đúng")

Tất cả đáp án đều đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đi zigzag theo hàng, lấy ô ở vị trí chẵn của đường đi)

```python
class Solution:
    def zigzagTraversal(self, grid: List[List[int]]) -> List[int]:
        ans = []
        k = 0                                          # vị trí trên đường đi (KHÔNG reset giữa các hàng)
        for i, row in enumerate(grid):
            for x in (row if i % 2 == 0 else reversed(row)):
                if k % 2 == 0:
                    ans.append(x)
                k += 1
        return ans
```

**Checklist**

- [ ] Hàng lẻ đã được đọc **ngược** (và không sửa `grid` gốc)?
- [ ] Bộ đếm/cờ nằm **ngoài** vòng lặp hàng (không reset mỗi hàng)?
- [ ] Lấy ô ở vị trí **chẵn** (đếm từ `0`): ô đầu tiên được lấy?
- [ ] Đã thử cả ma trận **số cột lẻ** (ví dụ `3 × 3`) và **số cột chẵn** (ví dụ `3 × 2`)?
- [ ] Độ dài kết quả bằng `(rows * cols + 1) // 2`?
- [ ] Trả về `list` các giá trị (không phải toạ độ) theo đúng thứ tự đi?